# Recuperación de información sobre BBC News

Notebook de entrega de la práctica. La lógica vive en `src/*.py`; aquí se importan y ejecutan esos scripts y se muestran las tablas y figuras de `reports/`.

Cada sección corresponde a una etapa del pipeline que ya pasó la validación de calidad (`reports/validacion/<etapa>.md`). Para reproducir el entorno, ver `ENTORNO.md`.

## 0. Setup

Estructura de carpetas, dependencias (`requirements.txt`) y entorno `bbc-ir` (Python 3.11). La celda siguiente ubica el notebook en la raíz del repositorio, agrega `src/` a la ruta de importación y comprueba las librerías.

In [1]:
import os, sys, json
from pathlib import Path
from importlib.metadata import version

# Raíz del repositorio: la carpeta que contiene src/ y estado/
RAIZ = Path.cwd()
while not (RAIZ / 'src').is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))

import pandas as pd
import spacy
from IPython.display import Image, Markdown, display

spacy.load('en_core_web_sm')  # el corpus está en inglés
pd.set_option('display.max_colwidth', 80)

libs = ['pandas', 'numpy', 'scikit-learn', 'scipy', 'rank-bm25', 'gensim', 'spacy', 'nltk',
        'matplotlib', 'wordcloud', 'python-pptx', 'pyyaml', 'joblib']
print('Python', sys.version.split()[0])
pd.DataFrame({'libreria': libs, 'version': [version(l) for l in libs]})

Python 3.11.17


,libreria,version
0,pandas,3.0.6
1,numpy,2.4.6
2,scikit-learn,1.9.1
3,scipy,1.17.1
4,rank-bm25,0.2.2
5,gensim,4.4.0
6,spacy,3.8.16
7,nltk,3.10.3
8,matplotlib,3.11.2
9,wordcloud,1.9.6


In [2]:
# Estado del pipeline (fuente de verdad: estado/progreso.json)
progreso = json.load(open('estado/progreso.json'))
pd.DataFrame(progreso['etapas']).T[['estado', 'intentos', 'commit']]

,estado,intentos,commit
setup,PASS,0,679174d
descarga,PASS,0,None
eda_raw,PENDIENTE,0,None
limpieza,PENDIENTE,0,None
eda_corpus,PENDIENTE,0,None
indices,PENDIENTE,0,None
evaluacion,PENDIENTE,0,None
presentacion,PENDIENTE,0,None


## 1. Descarga

`src/descarga.py` obtiene `data/raw/bbc-text.csv` (si ya existe no lo vuelve a bajar), calcula su SHA-256 y verifica 2,225 filas y las 5 categorías esperadas.

In [3]:
import descarga

df_raw = descarga.main()

ya existe: /Users/mikitl/Documents/FES/Ciencia de Datos/Clases/míneriaTextos/BBC-News-Classification/data/raw/bbc-text.csv (5,059,719 bytes)
SHA-256: e7652d6aa0fa92b2d02c6a992ebda82281041f83e7c2a92ae6e9c1c444981a8d
shape: (2225, 2)
conteo por categoría:
category
sport            511
business         510
politics         417
tech             401
entertainment    386
nulos por columna:
category    0
text        0
verificación OK


In [4]:
conteo = df_raw['category'].value_counts().rename_axis('categoria').reset_index(name='documentos')
conteo['porcentaje'] = (conteo['documentos'] / len(df_raw) * 100).map('{:.2f}%'.format)
print(f'Filas: {len(df_raw):,} | SHA-256:', open(descarga.RUTA_SHA).read().split()[0])
conteo

Filas: 2,225 | SHA-256: e7652d6aa0fa92b2d02c6a992ebda82281041f83e7c2a92ae6e9c1c444981a8d


,categoria,documentos,porcentaje
0,sport,511,22.97%
1,business,510,22.92%
2,politics,417,18.74%
3,tech,401,18.02%
4,entertainment,386,17.35%


In [5]:
display(Markdown(open('reports/descarga.md', encoding='utf-8').read()))

# Reporte de descarga

- **Fecha:** 2026-10-11
- **URL oficial:** https://raw.githubusercontent.com/mdsohaib/BBC-News-Classification/master/bbc-text.csv (repositorio: https://github.com/mdsohaib/BBC-News-Classification, rama `master`)
- **Origen usado:** copia local ya existente en `data/raw/bbc-text.csv` (mismo archivo del repositorio oficial). No se descargó de internet; el script tomó la rama `ya existe`.
- **Tamaño:** 5,059,719 bytes
- **SHA-256:** `e7652d6aa0fa92b2d02c6a992ebda82281041f83e7c2a92ae6e9c1c444981a8d` (guardado en `data/raw/bbc-text.sha256`)
- **Columnas:** `category`, `text`
- **Filas:** 2,225 (coincide con lo esperado)
- **Nulos:** 0 en `category`, 0 en `text`
- **Duplicados:** 99 textos duplicados (se eliminan en la etapa `limpieza`, no aquí)

## Conteo por categoría

| Categoría | Documentos |
|---|---:|
| sport | 511 |
| business | 510 |
| politics | 417 |
| tech | 401 |
| entertainment | 386 |
| **Total** | **2,225** |

## Verificación

Filas = 2,225 y las 5 categorías esperadas con los conteos esperados: **OK**. El texto no fue limpiado ni modificado.
